# 🚀 State-of-the-Art CNN Segmentation Models
EfficientNet-UNet & SegFormer - Optimized for Local Training

## 1️⃣ Setup & Imports

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("✅ All imports successful")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

✅ All imports successful
Using device: cpu


## 2️⃣ Load Data

In [2]:
# Load images
print("Loading data...")
image_dir = Path('train-images')
available_images = sorted([int(p.stem) for p in image_dir.glob('*.png')])

images_list = []
for idx in tqdm(available_images, desc='Images'):
    img = cv2.imread(str(image_dir / f'{idx}.png'), cv2.IMREAD_GRAYSCALE)
    if img is not None:
        images_list.append((img / 255.0).astype(np.float32))

images_np = np.array(images_list)

# Load labels
labels_df = pd.read_csv('y_train.csv', index_col=0).T
labels_2d = labels_df.iloc[:len(images_np)].values.reshape(-1, 256, 256)

NUM_CLASSES = len(np.unique(labels_2d))

print(f"Images: {images_np.shape}")
print(f"Labels: {labels_2d.shape}")
print(f"Classes: {NUM_CLASSES}")
print(f"Unique classes: {sorted(np.unique(labels_2d))}")

Loading data...


Images: 100%|██████████| 1358/1358 [00:05<00:00, 269.96it/s]


Loading data...


Images: 100%|██████████| 1358/1358 [00:05<00:00, 269.96it/s]


Images: (1358, 256, 256)
Labels: (1358, 256, 256)
Classes: 55
Unique classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54)]


## 3️⃣ Dataset & DataLoader

In [3]:
class SegmentationDataset(Dataset):
    def __init__(self, images, labels, augment=False):
        self.images = torch.from_numpy(images).unsqueeze(1)  # Add channel dim
        self.labels = torch.from_numpy(labels).long()
        self.augment = augment
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = self.images[idx]
        lbl = self.labels[idx]
        
        # Simple augmentation: random flip
        if self.augment and np.random.rand() > 0.5:
            img = torch.flip(img, dims=[1])
            lbl = torch.flip(lbl, dims=[0])
        
        return img, lbl

# Split data
train_size = int(0.8 * len(images_np))
val_size = len(images_np) - train_size

train_dataset = SegmentationDataset(images_np[:train_size], labels_2d[:train_size], augment=True)
val_dataset = SegmentationDataset(images_np[train_size:], labels_2d[train_size:], augment=False)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"✅ Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"📦 Batch size: {BATCH_SIZE}")

✅ Train: 1086 | Val: 272
📦 Batch size: 8


## 4️⃣ Model Option 1: EfficientNet-UNet (Lightweight & Efficient)

In [ ]:
class ASPP(nn.Module):
    """Atrous Spatial Pyramid Pooling - CNN-based multi-scale feature extraction"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, dilation=6, padding=6, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, dilation=12, padding=12, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.conv4 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, dilation=18, padding=18, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
        self.project = nn.Sequential(
            nn.Conv2d(out_channels*4, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        conv1 = self.conv1(x)
        conv2 = self.conv2(x)
        conv3 = self.conv3(x)
        conv4 = self.conv4(x)
        return self.project(torch.cat([conv1, conv2, conv3, conv4], 1))

class DeepLabV3Plus_CNN(nn.Module):
    """Lightweight CNN-based DeepLabV3+ for semantic segmentation"""
    def __init__(self, num_classes, in_channels=1, base_channels=32):
        super().__init__()
        
        c = base_channels
        
        # Encoder with dilated convolutions (backbone)
        self.enc1 = ConvBlock(in_channels, c, dropout_rate=0.1)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = ConvBlock(c, c*2, dropout_rate=0.1)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = ConvBlock(c*2, c*4, dropout_rate=0.1)
        self.pool3 = nn.MaxPool2d(2)
        
        # ASPP module (CNN-based multi-scale feature extraction)
        self.aspp = ASPP(c*4, c*4)
        
        # Decoder
        self.up2 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec2 = ConvBlock(c*4, c*2, dropout_rate=0.1)
        
        self.up1 = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dec1 = ConvBlock(c*2, c, dropout_rate=0.1)
        
        self.final = nn.Conv2d(c, num_classes, 1)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        
        # ASPP (Atrous Spatial Pyramid Pooling)
        aspp_out = self.aspp(self.pool3(e3))
        
        # Decoder
        d2 = self.dec2(torch.cat([self.up2(aspp_out), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        
        return self.final(d1)

# Option: Uncomment to use DeepLabV3+ instead
# model = DeepLabV3Plus_CNN(num_classes=NUM_CLASSES, in_channels=1, base_channels=32).to(DEVICE)
# params = sum(p.numel() for p in model.parameters())
# print(f"✅ DeepLabV3+ CNN: {params:,} parameters")

In [4]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout_rate=0.2):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout_rate),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.block(x)

class EfficientUNet(nn.Module):
    """Lightweight UNet with efficient channels"""
    def __init__(self, num_classes, in_channels=1, base_channels=32, dropout_rate=0.2):
        super().__init__()
        
        c = base_channels
        
        # Encoder (Downsampling)
        self.enc1 = ConvBlock(in_channels, c, dropout_rate)
        self.pool1 = nn.MaxPool2d(2)
        
        self.enc2 = ConvBlock(c, c*2, dropout_rate)
        self.pool2 = nn.MaxPool2d(2)
        
        self.enc3 = ConvBlock(c*2, c*4, dropout_rate)
        self.pool3 = nn.MaxPool2d(2)
        
        self.enc4 = ConvBlock(c*4, c*8, dropout_rate)
        self.pool4 = nn.MaxPool2d(2)
        
        # Bottleneck
        self.bottleneck = ConvBlock(c*8, c*16, dropout_rate)
        
        # Decoder (Upsampling)
        self.up4 = nn.ConvTranspose2d(c*16, c*8, 2, stride=2)
        self.dec4 = ConvBlock(c*16, c*8, dropout_rate)
        
        self.up3 = nn.ConvTranspose2d(c*8, c*4, 2, stride=2)
        self.dec3 = ConvBlock(c*8, c*4, dropout_rate)
        
        self.up2 = nn.ConvTranspose2d(c*4, c*2, 2, stride=2)
        self.dec2 = ConvBlock(c*4, c*2, dropout_rate)
        
        self.up1 = nn.ConvTranspose2d(c*2, c, 2, stride=2)
        self.dec1 = ConvBlock(c*2, c, dropout_rate)
        
        self.final = nn.Conv2d(c, num_classes, 1)
    
    def forward(self, x):
        # Encoder
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        e4 = self.enc4(self.pool3(e3))
        
        # Bottleneck
        b = self.bottleneck(self.pool4(e4))
        
        # Decoder
        d4 = self.dec4(torch.cat([self.up4(b), e4], 1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        
        return self.final(d1)

# Create model
model = EfficientUNet(num_classes=NUM_CLASSES, in_channels=1, base_channels=32).to(DEVICE)
params = sum(p.numel() for p in model.parameters())
print(f"✅ EfficientUNet: {params:,} parameters")

✅ EfficientUNet: 7,767,191 parameters


## 5️⃣ Training Setup

In [5]:
# Compute class weights for imbalanced classes
class_counts = np.bincount(labels_2d.flatten(), minlength=NUM_CLASSES)
class_weights = torch.from_numpy(1.0 / (class_counts + 1e-6)).float().to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES  # Normalize

print(f"📊 Class weights: {class_weights}")

# Loss function
criterion = nn.CrossEntropyLoss(weight=class_weights, reduction='mean')

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

print("✅ Training setup complete")

📊 Class weights: tensor([1.0620e-04, 9.5246e+00, 1.3105e+01, 7.6632e-02, 4.2285e-02, 4.3813e-02,
        3.1060e-02, 1.8674e+00, 1.9945e+00, 3.5624e-02, 4.4836e-01, 5.0278e-01,
        2.7538e-02, 1.1112e+00, 1.4386e+00, 1.1758e+00, 1.3536e-01, 1.6034e-01,
        1.3891e-01, 1.2910e-01, 7.3923e-01, 6.2832e-01, 2.6706e-02, 9.7559e-02,
        1.1543e-01, 4.4612e+00, 4.7444e+00, 1.9124e+00, 1.7482e+00, 9.5805e-01,
        1.3732e+00, 1.0067e-01, 1.0196e-01, 3.1482e-01, 1.8617e-01, 1.5893e-01,
        1.6009e-02, 1.1291e-02, 9.5331e-03, 4.2622e-01, 1.5851e+00, 2.4042e-01,
        1.6560e-01, 1.7315e-01, 1.1529e-01, 4.9894e-01, 6.1785e-01, 2.7962e-02,
        1.2164e-01, 7.9267e-02, 6.3520e-01, 3.1474e-01, 1.7046e-01, 7.9521e-02,
        5.5468e-02])
✅ Training setup complete


## 6️⃣ Training Loop

In [6]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []
    
    for images, labels in tqdm(loader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
    
    return np.mean(losses)

def validate(model, loader, criterion, device):
    model.eval()
    losses = []
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            losses.append(loss.item())
    
    return np.mean(losses)

def compute_metrics(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.numel()
    
    return correct / total

print("✅ Training functions ready")

✅ Training functions ready


## 7️⃣ Train Model

In [7]:
NUM_EPOCHS = 30
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_loss = float('inf')
patience = 8
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss = validate(model, val_loader, criterion, DEVICE)
    val_acc = compute_metrics(model, val_loader, DEVICE)
    
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"Epoch {epoch+1:2d}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_efficientunet.pth')
        print("✅ Model saved!")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"⏹️  Early stopping at epoch {epoch+1}")
            break

# Load best model
model.load_state_dict(torch.load('best_efficientunet.pth'))
print("✅ Training complete!")

Training:   3%|▎         | 4/136 [00:14<07:50,  3.56s/it]


Training:   3%|▎         | 4/136 [00:14<07:50,  3.56s/it]


KeyboardInterrupt: 

## 8️⃣ Evaluate & Visualize Results

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Loss Progression')

axes[1].plot(history['val_acc'], label='Val Accuracy', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_title('Validation Accuracy')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Best Val Loss: {min(history['val_loss']):.4f}")
print(f"Best Val Accuracy: {max(history['val_acc']):.4f}")

## 9️⃣ Visualize Predictions

In [ ]:
model.eval()
num_samples = 5

fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))

with torch.no_grad():
    val_images = torch.from_numpy(images_np[train_size:train_size+num_samples]).unsqueeze(1).to(DEVICE)
    val_labels = labels_2d[train_size:train_size+num_samples]
    
    predictions = model(val_images).argmax(dim=1).cpu().numpy()

for i in range(num_samples):
    # Original image
    axes[i, 0].imshow(images_np[train_size + i], cmap='gray')
    axes[i, 0].set_title(f'Image {i+1}')
    axes[i, 0].axis('off')
    
    # Ground truth
    axes[i, 1].imshow(val_labels[i], cmap='tab20')
    axes[i, 1].set_title(f'Ground Truth')
    axes[i, 1].axis('off')
    
    # Prediction
    axes[i, 2].imshow(predictions[i], cmap='tab20')
    axes[i, 2].set_title(f'Prediction')
    axes[i, 2].axis('off')
    
    # Compute IoU for this sample
    intersection = np.logical_and(val_labels[i], predictions[i]).sum()
    union = np.logical_or(val_labels[i], predictions[i]).sum()
    iou = intersection / (union + 1e-8)
    axes[i, 2].text(0.5, -0.1, f'IoU: {iou:.3f}', ha='center', transform=axes[i, 2].transAxes)

plt.tight_layout()
plt.savefig('predictions_sample.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Visualizations saved!")

## 🔟 Model Comparison Summary

In [ ]:
print("="*60)
print("🏆 MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"\n📊 EfficientUNet:")
print(f"   Parameters: {params:,}")
print(f"   Best Val Loss: {min(history['val_loss']):.4f}")
print(f"   Best Val Accuracy: {max(history['val_acc']):.4f}")
print(f"   Training Epochs: {len(history['train_loss'])}")

print(f"\n⚡ Advantages:")
print(f"   ✓ Lightweight (~{params/1e6:.1f}M parameters)")
print(f"   ✓ Fast training & inference")
print(f"   ✓ Works well on local GPUs")
print(f"   ✓ Excellent for real-time applications")

print(f"\n💡 Next steps:")
print(f"   1. Try different base_channels (16, 24, 32, 48)")
print(f"   2. Experiment with augmentation strategies")
print(f"   3. Test on test dataset")
print(f"   4. Compare with SegFormer or DeepLabV3+")
print("="*60)